In [ ]:
from utils import ReadImages, SplitViews, FeaturesExtraction, GetContours, StandardizeBubble, ReconstructionData, CreateFolder, SaveData
import pandas as pd

In [ ]:
# Load images from the specified path
images_path = "images/*.jpg"
img_array = ReadImages(images_path)

In [ ]:
# Split the images into two views
view_1, view_2 = SplitViews(img_array)

In [ ]:
%%capture
# Get contours for view 1 and save the boundaries in a folder
CreateFolder("images_results")
bubbles_1, grey_1 = GetContours(view_1, 200, view_name="view_1", folder="images_results")
bubbles_2, grey_2 = GetContours(view_2, 180, view_name="view_2", folder="images_results")

In [ ]:
# Synchronize bubble data in both views
bubbles_1, bubbles_2 = StandardizeBubble(bubbles_1, bubbles_2)

In [ ]:
%%capture
# Perform 3D reconstruction and save the results in a folder
Point_Cloud_3D, Volume, Surface_Area = ReconstructionData(bubbles_1, bubbles_2, folder="images_results", mm_per_pixel=0.076)

In [ ]:
# Extract features for ML model training
View_1_Features = FeaturesExtraction(bubbles_1, grey_1, 0.076)
View_2_Features = FeaturesExtraction(bubbles_2, grey_2, 0.076)

In [ ]:
View_1_Data = pd.concat([pd.DataFrame(View_1_Features), pd.DataFrame({"Volume": Volume, "Surface_Area": Surface_Area})], axis=1)
View_2_Data = pd.concat([pd.DataFrame(View_2_Features), pd.DataFrame({"Volume": Volume, "Surface_Area": Surface_Area})], axis=1)

In [ ]:
CreateFolder("data_results")
train_Data = pd.concat([View_1_Data, View_2_Data], ignore_index=True)
train_Data.to_csv("data_results/train_Data.csv", index=False)
train_Data

In [ ]:
# Perform a similar process for test data if available and save the results as test_Data.csv